# 🚀 Khởi động Dự án CrowdDiff trên Kaggle
Notebook này được thiết lập sẵn để chạy dự án **CrowdDiff** (Ước tính Mật độ Đám đông bằng Diffusion Models).  
Chạy từng ô theo thứ tự từ trên xuống dưới, hoặc nhấn **Run All**.

> ⚠️ **Lưu ý Quan Trọng:** Hãy chắc chắn rằng bạn đã bật GPU bằng cách vào mục **Accelerator** ở lề phải và chọn **GPU T4 x2** hoặc **GPU P100**.

## 📁 Bước 1: Nạp Mã Nguồn
Sao chép mã nguồn từ dataset đã upload lên Kaggle vào thư mục làm việc `working/`.

In [ ]:
!cp -r /kaggle/input/datasets/quangqunh/crowddiff-old/* /kaggle/working/
%cd /kaggle/working/
!ls

## 📦 Bước 2: Cài đặt Thư viện

In [ ]:
!pip install -q blobfile einops mpi4py
!pip install -q -r requirements_clean.txt
print('✅ Cài đặt thư viện hoàn tất!')

## 🗂️ Bước 3: Tạo Thư mục Kết quả

In [ ]:
import os
os.makedirs('/kaggle/working/results', exist_ok=True)
print('✅ Thư mục /kaggle/working/results đã sẵn sàng.')

## 🔍 Bước 4: Tìm File Model (.pt)
Ô này tự động tìm tất cả file model `.pt` trong toàn bộ Kaggle input.  
Dùng kết quả này để điền vào `MODEL_PATH` ở **Bước 6**.

In [ ]:
import glob

print('🔍 Đang tìm file model (.pt) trong /kaggle/input/ ...')
pt_files = glob.glob('/kaggle/input/**/*.pt', recursive=True)

if pt_files:
    print(f'\n✅ Tìm thấy {len(pt_files)} file model:')
    for f in pt_files:
        print(f'   {f}')
    print(f'\n👉 Dùng đường dẫn này trong Bước 6: MODEL_PATH = "{pt_files[-1]}"')
else:
    print('\n⚠️  Không tìm thấy file .pt nào!')
    print('   → Bạn cần TRAINING trước (Bước 5), hoặc')
    print('   → Upload file model lên Kaggle qua "Add Data".')

## 🔧 Bước 5: Chuẩn bị Dữ liệu (Pre-processing)
Hãy chắc chắn bạn đã tải **ShanghaiTech dataset** lên Kaggle qua **Add Data**.  
Chạy pre-processing cho cả **train** và **test**:

In [ ]:
# Pre-process tập TRAIN
!python cc_utils/preprocess_shtech.py \
    --data_dir /kaggle/input/shanghaitech/ShanghaiTech \
    --output_dir /kaggle/working \
    --dataset part_A \
    --mode train \
    --image_size 256 \
    --ndevices 1 \
    --sigma '0.5' \
    --kernel_size '3'
print('✅ Pre-process TRAIN xong!')

In [ ]:
# Pre-process tập TEST
!python cc_utils/preprocess_shtech.py \
    --data_dir /kaggle/input/shanghaitech/ShanghaiTech \
    --output_dir /kaggle/working \
    --dataset part_A \
    --mode test \
    --image_size 256 \
    --ndevices 1 \
    --sigma '0.5' \
    --kernel_size '3'
print('✅ Pre-process TEST xong!')

## 🏋️ Bước 6 (Tùy chọn): Huấn luyện (Training)
*(Nếu đã có model file `.pt` rồi — thấy ở Bước 4 — hãy **bỏ qua** bước này và chạy thẳng Bước 7.)*

In [ ]:
!PYTHONPATH="." CUDA_VISIBLE_DEVICES=0 python scripts/super_res_train.py \
    --data_dir /kaggle/working/part_A/part_1/train \
    --log_dir /kaggle/working/results \
    --resume_checkpoint /kaggle/input/datasets/quangqunh/weights/64_256_upsampler.pt \
    --normalizer 0.8 \
    --pred_channels 1 \
    --batch_size 1 \
    --attention_resolutions 32,16,8 \
    --class_cond False \
    --diffusion_steps 1000 \
    --large_size 256 \
    --small_size 256 \
    --learn_sigma True \
    --noise_schedule linear \
    --num_channels 192 \
    --num_head_channels 64 \
    --num_res_blocks 2 \
    --resblock_updown True \
    --use_fp16 True \
    --use_scale_shift_norm True

## 🧪 Bước 7: Kiểm thử (Testing / Inference)
⚠️ **Trước khi chạy:** Cập nhật `MODEL_PATH` bên dưới theo kết quả tìm được ở **Bước 4**.  

> 💡 Mỗi ảnh kết quả gồm: **ảnh gốc** | **ảnh dự đoán có overlay dấu ★ xanh** trên vị trí từng người.

In [ ]:
# ⚙️ CẬP NHẬT đường dẫn model tại đây (xem kết quả Bước 4)
MODEL_PATH = "/kaggle/working/results/ema_0.9999_030000.pt"

# Kiểm tra file model có tồn tại không
import os
if not os.path.exists(MODEL_PATH):
    import glob
    found = glob.glob('/kaggle/input/**/*.pt', recursive=True) + \
            glob.glob('/kaggle/working/results/*.pt')
    if found:
        MODEL_PATH = found[-1]
        print(f'⚠️  Model path gốc không tồn tại. Tự động dùng: {MODEL_PATH}')
    else:
        print('❌ Không tìm thấy file model! Hãy train trước (Bước 6) hoặc upload model.')
        raise FileNotFoundError('Model .pt file not found.')
else:
    print(f'✅ Sử dụng model: {MODEL_PATH}')

print('🚀 Bắt đầu inference...')

In [ ]:
!PYTHONPATH="." CUDA_VISIBLE_DEVICES=0 python scripts/super_res_sample.py \
    --data_dir /kaggle/working/part_A/part_1/test \
    --log_dir /kaggle/working/results \
    --model_path {MODEL_PATH} \
    --normalizer 0.8 \
    --pred_channels 1 \
    --batch_size 1 \
    --per_samples 1 \
    --attention_resolutions 32,16,8 \
    --class_cond False \
    --diffusion_steps 1000 \
    --large_size 256 \
    --small_size 256 \
    --learn_sigma True \
    --noise_schedule linear \
    --num_channels 192 \
    --num_head_channels 64 \
    --num_res_blocks 2 \
    --resblock_updown True \
    --use_fp16 True \
    --use_scale_shift_norm True

## 🖼️ Bước 8: Xem Kết quả Trực quan
Hiển thị các ảnh kết quả. Mỗi ảnh gồm:
- **Bên trái**: ảnh gốc
- **Bên phải**: ảnh có dấu ★ xanh đánh dấu vị trí đầu người được dự đoán

In [ ]:
import os
import glob
from PIL import Image
from matplotlib import pyplot as plt

result_dir = '/kaggle/working/results'
result_images = sorted(glob.glob(os.path.join(result_dir, '*.jpg')))

if not result_images:
    print('⚠️  Chưa có ảnh kết quả. Hãy chạy Bước 7 trước.')
else:
    print(f'✅ Tìm thấy {len(result_images)} ảnh kết quả. Hiển thị 5 ảnh đầu:')
    for path in result_images[:5]:
        name = os.path.basename(path)
        parts = name.replace('.jpg', '').split(' ')
        img_name   = parts[0]
        pred_count = parts[1] if len(parts) > 1 else '?'
        gt_count   = parts[2] if len(parts) > 2 else '?'

        img = Image.open(path)
        fig, ax = plt.subplots(1, 1, figsize=(14, 5))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(
            f'📷 {img_name}   |   Dự đoán: {pred_count}   |   Thực tế (GT): {gt_count}',
            fontsize=13, pad=10
        )
        plt.tight_layout()
        plt.show()
        print(f'  → {name}')